In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="transformers",
    notebook_name="02_multi_head_attention.ipynb"
)

# Multi-Head Attention: Looking at Things from Multiple Angles

In the [previous notebook](./01_attention_mechanisms.ipynb), we built single-head attention from scratch. But a single attention mechanism can only focus on **one type of relationship** at a time.

In this notebook, we'll:

1. Understand **why** we need multiple attention heads
2. Implement **multi-head attention** from scratch
3. **Visualize** what different heads learn to focus on
4. See how heads are **combined** into a final output

**Prerequisites:** Complete the [Attention Mechanisms notebook](./01_attention_mechanisms.ipynb) first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: Why Multiple Heads?

### Layer 1: The Expert Panel Analogy

Imagine you're making a big decision — whether to hire someone for a job. Instead of having one person evaluate everything, you bring in a panel of specialists:
- The technical expert asks: "Can they code?"
- The communication expert asks: "Can they explain ideas clearly?"
- The culture expert asks: "Do they work well in teams?"
- The project manager asks: "Do they hit deadlines?"

Each expert reads the same resume but asks different questions. At the end, you combine all their insights.

That's exactly what multi-head attention does. Each "head" is one specialist, asking its own question about the same input.

Consider: **"The tired cat sat on the soft mat."**

To understand "cat" fully, you need all of:
- What describes it? → "tired" (adjective relationship)
- What does it do? → "sat" (verb relationship)
- Where is it? → "on the mat" (location relationship)
- What kind? → "The" (determiner relationship)

One attention head can't focus on all four at once without blurring them together. Four heads, each specializing in one pattern, handle this cleanly.

### Layer 2: Why Heads Specialize During Training

At initialization, all heads are random and equivalent. How do they end up specialized? Through gradient descent and a symmetry-breaking process:

Early in training, different heads receive slightly different random gradients (because W_Q, W_K, W_V are initialized differently). One head might accidentally learn a pattern that helps reduce loss — say, connecting adjectives to their nouns. Once it does this well, the gradient for that head reinforces this pattern, while other heads are pushed to find OTHER patterns that still reduce loss.

This is analogous to biological neural networks developing specialization in visual cortex: V1 handles edges, V4 handles color, MT handles motion — not because they were programmed to, but because specialization reduced the training signal.

Research (Voita et al. 2019, Clark et al. 2019) has empirically categorized head types in trained BERT models:
- **Positional heads**: attend primarily to adjacent tokens
- **Syntactic heads**: connect grammatically related tokens (subject-verb, noun-determiner)
- **Semantic heads**: connect co-referring entities ("it" → "cat")
- **Separator heads**: often attend to punctuation or [SEP] tokens

## Part 2: Building Blocks (Reusing from Previous Notebook)

In [ ]:
def softmax(x):
    """Compute softmax along the last axis."""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Single-head scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    weights = softmax(scores)
    output = weights @ V
    return output, weights


print("Building blocks ready!")

## Part 3: Multi-Head Attention Step by Step

Here's the plan. You will build this in 4 stages:

```
1. Set up the embedding and decide: how many heads, what dimension per head?
2. For EACH head independently:
   a. Project input → smaller Q, K, V using this head's weight matrices
   b. Run the full attention computation (Q×K^T / √d_k → softmax → ×V)
3. Concatenate all head outputs back together
4. Mix them with one final projection matrix (W_O)
```

One thing to notice before the code: each head works with d_k = d_model / n_heads dimensions. This is how the total compute stays constant. More heads means smaller heads, not more compute.

In [ ]:
# Setup: 4-word sentence with 8-dimensional embeddings and 4 heads
sentence = ["The", "cat", "sat", "here"]
n_words = len(sentence)
d_model = 8     # Full embedding size
n_heads = 4     # Number of attention heads
d_k = d_model // n_heads  # Dimension per head = 8 / 4 = 2

print(f"Configuration:")
print(f"  Sentence:       {sentence}")
print(f"  d_model:        {d_model} (full embedding size)")
print(f"  n_heads:        {n_heads}")
print(f"  d_k (per head): {d_k} = d_model / n_heads = {d_model} / {n_heads}")
print(f"\nEach head works with {d_k}-dimensional Q, K, V")
print(f"  {n_heads} heads × {d_k} dims = {n_heads * d_k} = d_model (no information lost!)")

# Create input embeddings (random for demonstration)
X = np.random.randn(n_words, d_model)
print(f"\nInput X shape: {X.shape}  ({n_words} words × {d_model} dims)")

In [ ]:
# Step 1: Create weight matrices for EACH head
# Each head has its own W_Q, W_K, W_V

head_weights = []
for h in range(n_heads):
    W_Q_h = np.random.randn(d_model, d_k) * 0.3
    W_K_h = np.random.randn(d_model, d_k) * 0.3
    W_V_h = np.random.randn(d_model, d_k) * 0.3
    head_weights.append((W_Q_h, W_K_h, W_V_h))
    print(f"Head {h+1}: W_Q {W_Q_h.shape}, W_K {W_K_h.shape}, W_V {W_V_h.shape}")

print(f"\nEach head projects d_model={d_model} down to d_k={d_k}")

In [ ]:
# Step 2: Run attention INDEPENDENTLY in each head

head_outputs = []
head_attention_weights = []

for h in range(n_heads):
    W_Q_h, W_K_h, W_V_h = head_weights[h]
    
    # Project into this head's subspace
    Q_h = X @ W_Q_h  # (4 words × 2 dims)
    K_h = X @ W_K_h
    V_h = X @ W_V_h
    
    # Run attention
    output_h, weights_h = scaled_dot_product_attention(Q_h, K_h, V_h)
    
    head_outputs.append(output_h)
    head_attention_weights.append(weights_h)
    
    print(f"Head {h+1}: output shape = {output_h.shape}, "
          f"attention weights shape = {weights_h.shape}")

print(f"\nEach head produces a {n_words} × {d_k} output independently.")

In [ ]:
# Step 3: Concatenate all head outputs

concatenated = np.concatenate(head_outputs, axis=-1)  # Along last dimension

print("Concatenation:")
for h in range(n_heads):
    print(f"  Head {h+1} output: shape {head_outputs[h].shape}")
print(f"  {'─' * 40}")
print(f"  Concatenated:   shape {concatenated.shape}")
print(f"\n  {n_heads} heads × {d_k} dims per head = {concatenated.shape[1]} = d_model ✓")

In [ ]:
# Step 4: Final linear projection (W_O)
# This mixes information across heads

W_O = np.random.randn(d_model, d_model) * 0.3  # (8 × 8)

multi_head_output = concatenated @ W_O

print(f"Final projection:")
print(f"  Concatenated: {concatenated.shape}")
print(f"  × W_O:        {W_O.shape}")
print(f"  = Output:     {multi_head_output.shape}")
print(f"\nOutput has SAME shape as input ({n_words} × {d_model})")
print("This means multi-head attention can be stacked — output of one layer")
print("becomes the input of the next!")

## Part 4: Visualize What Each Head Focuses On

The most interesting part: **each head learns to focus on different things!**

Let's visualize the attention weights from all 4 heads side by side:

In [ ]:
fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))

for h in range(n_heads):
    ax = axes[h]
    w = head_attention_weights[h]
    
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(n_words))
    ax.set_yticks(range(n_words))
    ax.set_xticklabels(sentence, fontsize=11)
    ax.set_yticklabels(sentence, fontsize=11)
    ax.set_title(f'Head {h+1}', fontsize=14, fontweight='bold')
    
    if h == 0:
        ax.set_ylabel('Query (FROM)', fontsize=11)
    ax.set_xlabel('Key (TO)', fontsize=11)
    
    for i in range(n_words):
        for j in range(n_words):
            color = 'white' if w[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{w[i, j]:.2f}',
                    ha='center', va='center', fontsize=10, color=color)

fig.suptitle('Multi-Head Attention: Each Head Sees Different Patterns',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Notice how each head has a DIFFERENT attention pattern!")
print("In trained models, heads naturally specialize:")
print("  - Some focus on nearby words (positional)")
print("  - Some focus on grammar (syntactic)")
print("  - Some focus on meaning (semantic)")

## Part 5: The Complete Multi-Head Attention Function

The full formula is:

```
MultiHead(Q, K, V) = Concat(head₁, head₂, ..., headₕ) × W_O

where headᵢ = Attention(X × W_Qᵢ, X × W_Kᵢ, X × W_Vᵢ)
```

The code below wraps everything into one reusable class.

One design note: **W_O is not optional.** After concatenation, the output vector is just h separate blocks stacked end-to-end. Head 1's information sits in positions 0 to d_k-1, head 2's in d_k to 2d_k-1, etc. Without W_O, the heads are completely isolated — they can never combine their insights. W_O is the ONLY place in multi-head attention where different heads interact. It learns to ask: "given what all heads learned, what's the best combined representation?"

In [ ]:
class MultiHeadAttention:
    """
    Multi-Head Attention implemented from scratch with NumPy.
    
    Formula:
        MultiHead(Q, K, V) = Concat(head_1, ..., head_h) @ W_O
        where head_i = Attention(X @ W_Qi, X @ W_Ki, X @ W_Vi)
    """
    
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Dimension per head
        
        # Initialize weight matrices for each head
        scale = 0.3
        self.W_Q = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        
        # Output projection
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def forward(self, X, mask=None):
        """
        Run multi-head attention.
        
        Args:
            X: Input embeddings (num_words × d_model)
            mask: Optional attention mask
        
        Returns:
            output: Multi-head attention output (num_words × d_model)
            all_weights: Attention weights per head (list of num_words × num_words)
        """
        head_outputs = []
        all_weights = []
        
        for h in range(self.n_heads):
            # Project to this head's subspace
            Q_h = X @ self.W_Q[h]
            K_h = X @ self.W_K[h]
            V_h = X @ self.W_V[h]
            
            # Run attention
            out_h, w_h = scaled_dot_product_attention(Q_h, K_h, V_h, mask)
            head_outputs.append(out_h)
            all_weights.append(w_h)
        
        # Concatenate heads and project
        concatenated = np.concatenate(head_outputs, axis=-1)
        output = concatenated @ self.W_O
        
        return output, all_weights


# Test it!
mha = MultiHeadAttention(d_model=8, n_heads=4)

sentence = ["The", "cat", "sat", "here"]
X = np.random.randn(4, 8)

output, all_weights = mha.forward(X)

print(f"Input shape:  {X.shape}  ({len(sentence)} words × {X.shape[1]} dims)")
print(f"Output shape: {output.shape}  (same as input!)")
print(f"Number of attention weight matrices: {len(all_weights)}")
print(f"Each weight matrix shape: {all_weights[0].shape}")

## Part 6: Why Multi-Head Costs the Same as Single-Head

A natural worry: "Running h attention computations must be h times more expensive."

It isn't — because each head is h times smaller. Let's work through the exact math.

**Single-head attention** with d_model = 512:
- Q×K^T has cost proportional to n² × d_model

**Multi-head** with h=8 heads, each with d_k = 512/8 = 64:
- Each head's Q×K^T has cost proportional to n² × d_k = n² × 64
- 8 heads total: 8 × n² × 64 = n² × 512 = n² × d_model

**Identical.** Splitting d_model across h heads doesn't change the total multiply-accumulate operations — it just reorganizes them into h smaller, parallel chunks.

The same identity holds for:
- Q, K, V projection FLOPs: h × (d_model × d_k) = d_model² (same as single-head)
- W_O projection: d_model × d_model (additional vs single-head, but adds the cross-head interaction)

**Total per-layer FLOPs:** ~4 × n × d_model² (projections) + ~4 × n² × d_model (attention itself).

The code below verifies this numerically.

In [ ]:
d_model = 512
n_words = 10
n_heads_options = [1, 4, 8, 16]

print(f"d_model = {d_model}, n_words = {n_words}\n")
print(f"{'Heads':>6} {'d_k':>6} {'Ops per head':>15} {'Total attn ops':>15} {'Ratio':>8}")
print("─" * 55)

for n_h in n_heads_options:
    d_k = d_model // n_h
    # Main attention cost: Q×K^T is (n_words × d_k) @ (d_k × n_words) = n_words² × d_k
    ops_per_head = n_words * n_words * d_k
    total_ops = ops_per_head * n_h
    ratio = total_ops / (n_words * n_words * d_model)
    print(f"{n_h:>6} {d_k:>6} {ops_per_head:>15,} {total_ops:>15,} {ratio:>8.1f}x")

print(f"\nAll configurations have the SAME total computation!")
print(f"Splitting d_model across heads keeps cost constant.")
print(f"\nBonus: heads are independent, so they run in PARALLEL on a GPU.")

## Part 7: Real-World Head Specialization

Let's simulate what happens in trained models. We'll create heads that intentionally specialize in different patterns, to illustrate the concept:

In [ ]:
# Simulate specialized attention patterns from a trained model
sentence_long = ["The", "tired", "cat", "sat", "on", "the", "soft", "mat"]
n = len(sentence_long)

# Head 1: "Positional" — attends mostly to adjacent words
pos_weights = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        pos_weights[i, j] = np.exp(-abs(i - j) * 1.5)  # Exponential decay with distance
pos_weights = pos_weights / pos_weights.sum(axis=1, keepdims=True)

# Head 2: "Syntactic" — nouns attend to their determiners/adjectives
syn_weights = np.ones((n, n)) * 0.02  # Small baseline
syn_weights[2, 0] = 0.4   # "cat" → "The" (determiner)
syn_weights[2, 1] = 0.5   # "cat" → "tired" (adjective)
syn_weights[7, 5] = 0.4   # "mat" → "the" (determiner)
syn_weights[7, 6] = 0.5   # "mat" → "soft" (adjective)
syn_weights[3, 2] = 0.7   # "sat" → "cat" (subject)
syn_weights = syn_weights / syn_weights.sum(axis=1, keepdims=True)

# Head 3: "Semantic" — attends to related meaning
sem_weights = np.ones((n, n)) * 0.05
sem_weights[2, 7] = 0.4   # "cat" ↔ "mat" (rhyme, both nouns)
sem_weights[7, 2] = 0.4   # "mat" ↔ "cat"
sem_weights[1, 6] = 0.4   # "tired" ↔ "soft" (both adjectives)
sem_weights[6, 1] = 0.4   # "soft" ↔ "tired"
sem_weights[3, 4] = 0.3   # "sat" → "on" (verb-preposition)
sem_weights = sem_weights / sem_weights.sum(axis=1, keepdims=True)

# Head 4: "Long-range" — attends to distant but related words
lr_weights = np.ones((n, n)) * 0.04
lr_weights[3, 7] = 0.5   # "sat" → "mat" (where did it sit?)
lr_weights[4, 7] = 0.5   # "on" → "mat" (on what?)
lr_weights[7, 3] = 0.4   # "mat" → "sat" (what happened on me?)
lr_weights[7, 4] = 0.3   # "mat" → "on"
lr_weights = lr_weights / lr_weights.sum(axis=1, keepdims=True)

# Visualize all four
head_names = ["Head 1: Positional\n(nearby words)",
              "Head 2: Syntactic\n(grammar)",
              "Head 3: Semantic\n(meaning)",
              "Head 4: Long-range\n(distant relations)"]
all_patterns = [pos_weights, syn_weights, sem_weights, lr_weights]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for h, (ax, w, name) in enumerate(zip(axes, all_patterns, head_names)):
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.6)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(sentence_long, fontsize=9, rotation=45, ha='right')
    ax.set_yticklabels(sentence_long, fontsize=9)
    ax.set_title(name, fontsize=12, fontweight='bold')
    if h == 0:
        ax.set_ylabel('FROM', fontsize=11)

fig.suptitle('How Different Heads Specialize in Trained Models',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("In real trained models, heads naturally develop these specializations!")
print("Nobody programs them — the model discovers useful patterns during training.")

## Part 8: Average vs. Multi-Head

Let's show what happens if we just average all heads vs. keeping them separate:

In [ ]:
# Average all head patterns
avg_pattern = np.mean(all_patterns, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show averaged pattern
im1 = axes[0].imshow(avg_pattern, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
axes[0].set_xticks(range(n))
axes[0].set_yticks(range(n))
axes[0].set_xticklabels(sentence_long, fontsize=10, rotation=45, ha='right')
axes[0].set_yticklabels(sentence_long, fontsize=10)
axes[0].set_title('Single Head (averaged)\nAll patterns blurred together', fontsize=13)

# Show one specialized head
im2 = axes[1].imshow(syn_weights, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
axes[1].set_xticks(range(n))
axes[1].set_yticks(range(n))
axes[1].set_xticklabels(sentence_long, fontsize=10, rotation=45, ha='right')
axes[1].set_yticklabels(sentence_long, fontsize=10)
axes[1].set_title('Specialized Head (syntactic)\nClear grammar patterns', fontsize=13)

plt.tight_layout()
plt.show()

print("The averaged pattern loses the sharp, specialized patterns.")
print("Multi-head attention preserves each head's unique expertise.")
print("The final projection (W_O) learns to combine them intelligently.")

## Part 9: Common Model Configurations

Let's look at how real models configure their attention:

In [ ]:
models = [
    ("GPT-2 Small",   768,   12, 117),
    ("BERT Base",      768,   12, 110),
    ("GPT-2 Medium",  1024,  16, 345),
    ("BERT Large",    1024,  16, 340),
    ("GPT-2 Large",   1280,  20, 774),
    ("GPT-3",        12288,  96, 175000),
    ("LLaMA 7B",      4096,  32, 7000),
    ("LLaMA 70B",     8192,  64, 70000),
]

print(f"{'Model':<16} {'d_model':>8} {'Heads':>6} {'d_k':>6} {'Params':>10}")
print("─" * 50)
for name, d_m, heads, params in models:
    d_k = d_m // heads
    p_str = f"{params}M" if params < 1000 else f"{params/1000:.0f}B"
    print(f"{name:<16} {d_m:>8} {heads:>6} {d_k:>6} {p_str:>10}")

print(f"\nPattern: d_k is typically 64 or 128 across all model sizes.")
print(f"As models get bigger, they add more heads rather than bigger heads.")

## Summary

### The Multi-Head Attention Formula

```
MultiHead(Q, K, V) = Concat(head₁, head₂, ..., headₕ) × W_O

where headᵢ = Attention(X × W_Qᵢ, X × W_Kᵢ, X × W_Vᵢ)
```

### Key Takeaways

1. **Multiple heads** capture different types of relationships (grammar, meaning, position)
2. Each head works in a **smaller dimension** (d_model / n_heads)
3. Heads run **independently and in parallel** — no extra cost!
4. **Concatenation + W_O** combines all heads' insights
5. Heads **specialize naturally** during training — nobody programs them
6. Output shape **equals input shape** — so layers can stack

**Next:** [Positional Encoding](./03_positional_encoding.ipynb) — how transformers know word order.

## Layer 2: Expert Depth

### Parameter Count Formula (Derived)

For one multi-head attention block with d_model dimensions and h heads (d_k = d_model/h):

| Component | Shape | Parameters |
|-----------|-------|------------|
| W_Qᵢ per head | d_model × d_k | d_model × d_k |
| W_Kᵢ per head | d_model × d_k | d_model × d_k |
| W_Vᵢ per head | d_model × d_k | d_model × d_k |
| All Q, K, V (h heads) | — | 3 × h × d_model × d_k = 3 × d_model² |
| W_O | d_model × d_model | d_model² |
| **Total** | — | **4 × d_model²** |

The h (number of heads) cancels out completely. The total is always 4 × d_model², regardless of how many heads you use.

Example: BERT Base (d_model = 768): 4 × 768² = 2,359,296 ≈ 2.36M per attention block. 12 layers: 12 × 2.36M = 28.3M parameters just in attention.

### Failure Modes

**Head collapse**: multiple heads learn the same attention pattern. This wastes capacity. Diagnosis: compute pairwise cosine similarity between all heads' attention weight matrices on a validation set. If pairs have similarity > 0.8, those heads have collapsed. Fix: attention dropout (0.1 is standard), which randomly zeros attention weights during training, forcing other heads to compensate. Also ensures diverse initialization scaling.

**Single head monopolizing W_O**: one head can dominate the W_O projection by learning a much larger weight norm for its slice of the output. The other heads contribute near-zero signal. Detect by examining the per-head row norm of W_O (partition the d_model rows into h groups). Fix: weight decay during training; explicitly regularizing per-head W_O norms.

**Batched masking bugs**: when processing batches of sequences with different lengths, padding must be masked. Common bug: mask shape (batch_size, seq_len) vs (batch_size, 1, 1, seq_len) — wrong broadcasting leaves some padded positions unmasked. Always verify with a batch where the second sequence is shorter and check that padded positions produce zero attention weight.

### Staff/Principal Interview Q&A

**Q: Why is W_O necessary? Prove you can't just sum or average head outputs.**

Without W_O: output[position] = Concat(head₁[pos], ..., headₕ[pos]). In this concatenation, position d in the output comes from exactly one head. No head knows what the other heads computed. For example, head 1 (syntactic) and head 3 (semantic) cannot combine their findings. W_O is a (d_model × d_model) matrix. When you multiply by it, each output dimension becomes a linear combination of ALL head contributions. This is the only cross-head information mixing in the entire attention block. Summing head outputs (instead) would lose the identity of which head contributed what — the model couldn't learn "combine feature from head 2 WITH feature from head 5." Averaging is equivalent to learning W_O with the constraint that all per-head blocks are the same — strictly less expressive.

**Q: Explain Grouped Query Attention (GQA). Why does it matter for production inference?**

In standard MHA with n_q query heads, each head has its own K and V projections: n_kv_heads = n_q. During autoregressive inference, you cache all K and V tensors for past tokens. Memory cost: 2 × n_layers × n_q × d_k × n_tokens × bytes_per_element. For LLaMA-2 70B (80 layers, 64 heads, d_k=128, float16): 2 × 80 × 64 × 128 × n_tokens × 2 bytes ≈ 5.3GB for 4096 tokens per concurrent request. At 100 concurrent users, that's 530GB just for KV cache. GQA (Ainslie et al. 2023): use n_kv_heads < n_q. Groups of query heads share one K,V pair. LLaMA-2 70B uses n_kv_heads = 8 (vs n_q = 64) — an 8× KV cache reduction, from 5.3GB to 665MB per request. Quality drop is small (< 1% on most benchmarks) because different query heads asking similar questions can share a single K,V memory. MQA (n_kv_heads = 1) reduces cache even more but has noticeable quality loss.

**Q: How would you diagnose whether multi-head attention is the bottleneck in a slow transformer inference pipeline?**

Step 1: Profile with torch.profiler or CUDA nsys to measure time per operation. Step 2: Check if most time is in `baddbmm` (batch matrix multiply — the QK^T step) vs `linear` (projections). If QK^T dominates, you have a long-context problem and should look at Flash Attention, sparse attention, or shorter context. If projections dominate, you're likely bottlenecked by weight loading from memory, which suggests quantization (reduce bytes per weight) or tensor parallelism (split W_O across GPUs). Step 3: Check memory bandwidth utilization — transformers at inference are often memory-bandwidth-bound, not compute-bound, especially for small batch sizes. If GPU utilization is < 30% but memory bandwidth is near saturation, the fix is batching more requests together, not faster attention math.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)